# Set the microscope limits

How far the stage and the focus drive may travel, and what automation may choose. A range includes both ends; `[]` means reviewed and unrestricted. Every range is cut to what NIS-Elements itself allows when the driver connects, so a limit can only ever narrow the travel.

## Configure

Review the starting limits. The Measure step replaces x and y with the rectangle spanned by the corners you record; type the safe z range yourself, since it depends on the sample and the objective.

In [ ]:
import bootstrap

from zmart_drivers.nikon.nis_elements_ar610.driver_setup.limits.config import (
    adopt_limits,
    validate_limits,
)

LIMITS = {
    "x_um": {"range": [-50000, 50000]},
    "y_um": {"range": [-30000, 30000]},
    "z_um": {"range": [0, 9000]},
    "objective_slot": [],
    "optical_configuration": [],
    "exposure_ms": {"range": [0.01, 10000]},
    "autofocus_range_um": {"range": [1, 500]},
}

## Measure

Record the corners of the safe travel. Drive the stage to each corner with the joystick or in NIS-Elements, with the objective clear of everything, then run that corner's cell; it reads where the stage stands. To record a corner again, move the stage and run its cell again.

In [ ]:
from zmart_drivers.nikon import nis_elements_ar610 as nis_elements
from zmart_drivers.nikon.nis_elements_ar610.driver_setup.limits.measure_limits import (
    CornerRecorder,
)

scope = nis_elements.connect(load_calibration=False)
corners = CornerRecorder(scope.client)

**Corner 1:** drive the stage to the first safe corner, then run the cell below.

In [ ]:
corners.record(1)

**Corner 2:** drive the stage to the opposite corner, then run the cell below.

In [ ]:
corners.record(2)

**Corners 3 and 4** (optional): the other two corners, for a holder that is not square to the stage.

In [ ]:
corners.record(3)

In [ ]:
corners.record(4)

The x and y limits spanned by the recorded corners.

In [ ]:
captured_xy = corners.limits()
LIMITS.update(captured_xy["limits"])
print("x:", LIMITS["x_um"]["range"])
print("y:", LIMITS["y_um"]["range"])

## Validate

Check the complete limits document before saving it.

In [ ]:
validated_limits = validate_limits(LIMITS)

## Save and adopt

This saves the completed notebook, checks that its measurement cells ran, then publishes the limits together with the notebook, which records the corners. The driver loads them at the next connect.

In [ ]:
published = bootstrap.NOTEBOOK.save_and_adopt(
    lambda notebook: adopt_limits(validated_limits, notebook_paths=[notebook])
)
published